# Notebook for downloading and formatting TSMO data

In [ ]:
import pandas as pd
import requests
from arcgis import GIS
from arcgis.features import FeatureLayer
from utils import *
import arcpy
import os


## Caltrans Cameras and Signs

In [ ]:
camera_link = 'https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CCTV/FeatureServer/0'
TSMO_Project = r"F:\GIS\PROJECTS\Transportation\TSMO\TSMO.gdb"

In [ ]:
camera_data = get_fs_data(camera_link)

In [ ]:
# write camera data to geodatabase
table_path = os.path.join(TSMO_Project, 'Caltrans_Cameras')
df_to_gdb_table(camera_data, table_path)
print(f"Table written: {table_path}")

In [ ]:
# convert table to point feature class using Latitude/Longitude
out_fc = os.path.join(TSMO_Project, 'Caltrans_Camera_Locations')
arcpy.management.XYTableToPoint(
    in_table=table_path,
    out_feature_class=out_fc,
    x_field='Longitude',
    y_field='Latitude',
    coordinate_system=arcpy.SpatialReference(4326)
)
print(f"Feature class created: {out_fc}")

In [ ]:
caltrans_message_signs = pd.read_csv('data/cmsStatusD03.csv')

In [ ]:
# write sign data to geodatabase
table_path = os.path.join(TSMO_Project, 'Caltrans_Message_Signs')
df_to_gdb_table(caltrans_message_signs, table_path)
print(f"Table written: {table_path}")

In [ ]:
# convert table to point feature class using Latitude/Longitude
out_fc = os.path.join(TSMO_Project, 'Caltrans_Message_Sign_Locations')
arcpy.management.XYTableToPoint(
    in_table=table_path,
    out_feature_class=out_fc,
    x_field='Longitude',
    y_field='Latitude',
    coordinate_system=arcpy.SpatialReference(4326)
)
print(f"Feature class created: {out_fc}")

# NDOT Downloads

In [ ]:
ndot_api_key = "9a16cc4cc5e2453f88998d5f9d6c897e"
cameras_url = f"https://www.nvroads.com/api/v2/get/cameras?key={ndot_api_key}&format=json"
roads_url = f"https://www.nvroads.com/api/v3/get/roadconditions?key={ndot_api_key}&format=json"
message_sign_url = f"https://www.nvroads.com/api/v2/get/messagesigns?key={ndot_api_key}&format=json"

## NDOT Cameras

In [ ]:
df_ndot_cameras = pd.DataFrame(requests.get(cameras_url).json())
df_ndot_cameras['Views'] = df_ndot_cameras['Views'].astype(str)

table_path = os.path.join(TSMO_Project, 'NDOT_Cameras')
df_to_gdb_table(df_ndot_cameras, table_path)
print(f"Table written: {table_path}")

out_fc = os.path.join(TSMO_Project, 'NDOT_Camera_Locations')
arcpy.management.XYTableToPoint(
    in_table=table_path,
    out_feature_class=out_fc,
    x_field='Longitude',
    y_field='Latitude',
    coordinate_system=arcpy.SpatialReference(4326)
)
print(f"Feature class created: {out_fc}")

## NDOT Road Conditions

In [ ]:
road_data = requests.get(roads_url).json()

out_fc = os.path.join(TSMO_Project, 'NDOT_Road_Conditions')
if arcpy.Exists(out_fc):
    arcpy.management.Delete(out_fc)

gdb, fc_name = os.path.split(out_fc)
arcpy.management.CreateFeatureclass(gdb, fc_name, 'POLYLINE', spatial_reference=arcpy.SpatialReference(4326))
arcpy.management.AddField(out_fc, 'road_id', 'LONG')
arcpy.management.AddField(out_fc, 'LocationDescription', 'TEXT', field_length=255)
arcpy.management.AddField(out_fc, 'OverallStatus', 'TEXT', field_length=100)
arcpy.management.AddField(out_fc, 'SecondaryConditions', 'TEXT', field_length=500)
arcpy.management.AddField(out_fc, 'AreaName', 'TEXT', field_length=100)
arcpy.management.AddField(out_fc, 'RoadwayName', 'TEXT', field_length=100)
arcpy.management.AddField(out_fc, 'LastUpdated', 'LONG')

fields = ['SHAPE@', 'road_id', 'LocationDescription', 'OverallStatus',
          'SecondaryConditions', 'AreaName', 'RoadwayName', 'LastUpdated']

with arcpy.da.InsertCursor(out_fc, fields) as cursor:
    for rec in road_data:
        parts = arcpy.Array()
        for encoded in rec.get('EncodedPolyline', []):
            coords = decode_polyline(encoded)
            parts.add(arcpy.Array([arcpy.Point(lon, lat) for lat, lon in coords]))
        geom = arcpy.Polyline(parts, arcpy.SpatialReference(4326)) if len(parts) > 0 else None
        cursor.insertRow([
            geom,
            rec.get('Id'),
            rec.get('LocationDescription', ''),
            rec.get('Overall Status', ''),
            str(rec.get('Secondary Conditions', [])),
            rec.get('AreaName', ''),
            rec.get('RoadwayName', ''),
            rec.get('LastUpdated')
        ])

print(f"Feature class created: {out_fc}")

## NDOT Message Signs

In [ ]:
df_ndot_signs = pd.DataFrame(requests.get(message_sign_url).json())
df_ndot_signs['Messages'] = df_ndot_signs['Messages'].astype(str)

table_path = os.path.join(TSMO_Project, 'NDOT_Message_Signs')
df_to_gdb_table(df_ndot_signs, table_path)
print(f"Table written: {table_path}")

out_fc = os.path.join(TSMO_Project, 'NDOT_Message_Sign_Locations')
arcpy.management.XYTableToPoint(
    in_table=table_path,
    out_feature_class=out_fc,
    x_field='Longitude',
    y_field='Latitude',
    coordinate_system=arcpy.SpatialReference(4326)
)
print(f"Feature class created: {out_fc}")

# Cell Coverage

In [ ]:
data_folder = r'F:\GIS\PROJECTS\Transportation\TSMO\data'
trpa_bndy =r"F:\GIS\PROJECTS\Transportation\TSMO\TSMO.gdb\TRPABoundary"


In [ ]:
import glob
arcpy.env.workspace = TSMO_Project

shapefiles = glob.glob(os.path.join(data_folder, '**', '*.shp'), recursive=True)

for shp in shapefiles:
    shp_name = os.path.splitext(os.path.basename(shp))[0]
    out_name = shp_name.split('_h3_')[0]
    out_fc = os.path.join(TSMO_Project, out_name)

    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    lyr = arcpy.management.MakeFeatureLayer(shp, shp_name).getOutput(0)

    arcpy.management.SelectLayerByLocation(
        in_layer=lyr,
        overlap_type="INTERSECT",
        select_features=trpa_bndy,
        selection_type="NEW_SELECTION"
    )

    arcpy.conversion.ExportFeatures(
        in_features=lyr,
        out_features=out_fc,
        where_clause="",
        use_field_alias_as_name="NOT_USE_ALIAS",
        sort_field=None
    )
    print(f"Exported: {out_fc}")

In [ ]:
print(shapefiles)

In [ ]:
shapefiles = glob.glob(os.path.join(data_folder, '**', '*.shp'), recursive=True)
#remove bdc_06_4GLTE_mobile_broadband
shapefiles = ['F:\\GIS\\PROJECTS\\Transportation\\TSMO\\data\\bdc_32_4GLTE_mobile_broadband_h3_J25_19mar2026\\bdc_32_4GLTE_mobile_broadband_h3_J25_19mar2026.shp', 'F:\\GIS\\PROJECTS\\Transportation\\TSMO\\data\\bdc_32_5GNR_mobile_broadband_h3_J25_19mar2026\\bdc_32_5GNR_mobile_broadband_h3_J25_19mar2026.shp']
for shp in shapefiles:
    shp_name = os.path.splitext(os.path.basename(shp))[0]
    fc_name = shp_name.split('_h3_')[0]
    fc = os.path.join(TSMO_Project, fc_name)
    print(f"Processing: {fc_name}")
    # state code from fc name: bdc_XX → chars at index 4-5
    state_code = fc_name[4:6]

    # get first value from technology column
    with arcpy.da.SearchCursor(fc, ['technology']) as cursor:
        tech_val = str(int(next(cursor)[0]))

    minup_field   = f'technology_{tech_val}_minup'
    mindown_field = f'technology_{tech_val}_mindown'

    arcpy.management.AddField(fc, minup_field,   'DOUBLE')
    arcpy.management.AddField(fc, mindown_field, 'DOUBLE')
    arcpy.management.AddField(fc, 'GridID', 'TEXT', field_length=50)

    arcpy.management.CalculateField(fc, minup_field,   '!minup!',   'PYTHON3')
    arcpy.management.CalculateField(fc, mindown_field, '!mindown!', 'PYTHON3')
    arcpy.management.CalculateField(fc, 'GridID', f'"{state_code}_" + str(!OBJECTID!)', 'PYTHON3')

    print(f"Updated: {fc_name}  |  tech={tech_val}  |  state={state_code}")

In [ ]:
from functools import reduce

arcpy.env.workspace = TSMO_Project
drop_fields = {'objectid', 'shape', 'environmnt', 'h3_res9_id', 'shape_length', 'shape_area',
               'technology', 'mindown', 'minup'}

dfs_06, dfs_32 = [], []

for fc_name in arcpy.ListFeatureClasses('bdc*'):
    fc = os.path.join(TSMO_Project, fc_name)
    fields = [f.name for f in arcpy.ListFields(fc) if f.name.lower() not in drop_fields]
    with arcpy.da.SearchCursor(fc, fields) as cursor:
        df = pd.DataFrame(list(cursor), columns=fields)
    if '_06_' in fc_name:
        dfs_06.append(df)
    elif '_32_' in fc_name:
        dfs_32.append(df)

df_06 = reduce(lambda l, r: pd.merge(l, r, on='GridID', how='outer'), dfs_06)
df_32 = reduce(lambda l, r: pd.merge(l, r, on='GridID', how='outer'), dfs_32)

df_combined = pd.concat([df_06, df_32], ignore_index=True)

table_path = os.path.join(TSMO_Project, 'Cell_Coverage')
df_to_gdb_table(df_combined, table_path)
print(f"Table written: {table_path}")
print(f"Rows: {len(df_combined):,}  |  Columns: {list(df_combined.columns)}")